# **Case 3: Cantilever Plate Beam Benchmark**

See [1_formulation.ipynb](1_formulation.ipynb) for the formulation derivation and the table of plate elements compared.


In [ ]:
from typing import Callable

import numpy as np
import matplotlib.pyplot as plt

import pyck as ck

FORMULATIONS = {
    "KL-1": ck.PlateKirchhoffLove1p,
    "RM-3": ck.PlateReissnerMindlin3p,
    "RM-D3": ck.PlateReissnerMindlinDispl3p,
    "RM-D2": ck.PlateReissnerMindlinDispl2p,
    "RM-D1": ck.PlateReissnerMindlin1p
}

In [ ]:
def max_displacement(problem, solution, n_eval=81):
    """Compute the maximum reconstructed transverse displacement."""
    plate = problem.patches[0]
    element = problem.element
    u = np.asarray(solution)[:problem.num_physical_dofs]

    grid = np.linspace(0.0, 1.0, n_eval)
    uu, vv = np.meshgrid(grid, grid, indexing="xy")
    pts = np.column_stack([uu.ravel(), vv.ravel()])

    shape = ck.eval_shape_at(plate, pts, order=2)
    Nw = element._cpp_object.displacement_shape_matrix(shape)
    w = Nw @ u

    return float(np.max(np.abs(w)))


def center_displacement(problem, solution):
    """Compute the reconstructed transverse displacement at the plate center."""
    plate = problem.patches[0]
    element = problem.element
    u = np.asarray(solution)[:problem.num_physical_dofs]

    shape = ck.eval_shape_at(plate, np.array([[0.5, 0.5]]), order=2)
    Nw = element._cpp_object.displacement_shape_matrix(shape)
    return float(abs((Nw @ u)[0]))


def strain_energy(solution, K):
    """Compute the strain energy from the solution and stiffness matrix."""
    num_phys = K.shape[0]
    u_phys = solution[:num_phys]
    return 0.5 * np.dot(u_phys, K @ u_phys)


def external_work_energy(solution, F):
    """Compute the energy from the physical external load vector."""
    u_phys = np.asarray(solution)[:F.shape[0]]
    return 0.5 * float(np.dot(F, u_phys))

In [ ]:
# Material constants reused from Case 1
E, nu = 10.92, 0.3

# Penalty scaling for clamped BC enforcement (matches Case 1)
PENALTY_SCALE = 1.0e6

# Above this slenderness ratio, penalty BCs are used; below, Lagrange.
THIN_PLATE_SWITCH = 100.0


## **Case 3: Cantilever Plate Beam Benchmark**

We now use an elongated cantilever plate as a beam-like benchmark. The plate is clamped at $X=0$ and loaded by a uniform transverse surface load $q$. For a sufficiently narrow plate, the centerline response should approach the Reissner-Mindlin/Timoshenko beam solution

$$
w(x,y) = \frac{1}{\alpha G A} \left( \overline{p} L x - \frac{\overline{p} x^2}{2} \right) - \frac{1}{EI} \left( - \frac{\overline{p} x^4}{24} + \frac{\overline{p} L x^3}{6} - \frac{\overline{p} L^2 x^2}{4} \right),
$$

with $A = B h$, $I = B h^3/12$, $G = E/[2(1 + \nu)]$, and $\overline{p} = |q| B$. The first term is the shear contribution and the second term is the bending contribution. The sweep below compares the numerical free-tip centerline displacement against this analytical value.

### **Finite-Width Plate vs. 1D Beam**

The Timoshenko beam reference assumes free lateral edges and an unconstrained cross-section; in a 2D plate the clamp at $X=0$ also suppresses the in-plane lateral Poisson contraction near the root, which slightly stiffens the response. The discrepancy is governed by $B/L$: it vanishes as $B/L \to 0$, and grows toward the cylindrical-bending limit $1-\nu^2$ as $B/L$ becomes large. To stay close to the 1D beam reference, we use a slender ratio $B/L = 0.02$, which keeps the residual offset below $0.4\%$ for the locking-free formulations.

### **Slenderness Range**

The sweep is capped at $L/h = 10^6$. The standard `RM-3` formulation suffers from shear locking at high slenderness, and beyond this point the linear system becomes severely ill-conditioned: the tip displacement no longer decreases monotonically but oscillates erratically (including non-physical overshoots above unity) due to numerical noise rather than meaningful behaviour. Truncating the sweep keeps the locking trend visible without polluting the comparison with conditioning artefacts.

The boundary enforcement follows the same rule as the clamped-plate sweep in section 1.1: Lagrange multipliers are used in the thick regime, and penalty constraints are used after the thin-plate switch.


In [ ]:
CANTILEVER_LENGTH = 10.0
CANTILEVER_WIDTH = 0.2
CANTILEVER_PDEG = 3
CANTILEVER_NBASIS_U = 80
CANTILEVER_NBASIS_V = 4
CANTILEVER_LOAD = -1.0
CANTILEVER_SHEAR_CORRECTION = 5.0 / 6.0
CANTILEVER_SLENDERNESS_VALUES = np.logspace(np.log10(2.0), np.log10(5.0e5), 13)
CANTILEVER_SHAPE_SLENDERNESS_TARGET = 10.0
CANTILEVER_SHAPE_INDEX = int(
    np.argmin(np.abs(CANTILEVER_SLENDERNESS_VALUES - CANTILEVER_SHAPE_SLENDERNESS_TARGET))
)
CANTILEVER_SHAPE_SLENDERNESS = CANTILEVER_SLENDERNESS_VALUES[CANTILEVER_SHAPE_INDEX]
CANTILEVER_LEGEND_LABELS = {
    "KL-1": "KL-1 (standard)",
    "RM-3": "RM-3 (standard)",
    "RM-D3": "RM-D3 (Oesterle et al.)",
    "RM-D2": "RM-D2 (this study)",
    "RM-D1": "RM-D1 (Tran et al.)",
}


def cantilever_load(pts):
    return CANTILEVER_LOAD * np.ones(pts.shape[0])


def rm_beam_cantilever_components(x, length, width, thickness, load, E, nu):
    x = np.asarray(x, dtype=float)
    p_bar = abs(load) * width
    area = width * thickness
    inertia = width * thickness**3 / 12.0
    shear_modulus = E / (2.0 * (1.0 + nu))

    shear = (p_bar * length * x - 0.5 * p_bar * x**2) / (
        CANTILEVER_SHEAR_CORRECTION * shear_modulus * area
    )
    bending = -(
        -p_bar * x**4 / 24.0
        + p_bar * length * x**3 / 6.0
        - p_bar * length**2 * x**2 / 4.0
    ) / (E * inertia)
    return shear, bending


def rm_beam_cantilever_deflection(x, length, width, thickness, load, E, nu):
    shear, bending = rm_beam_cantilever_components(
        x, length, width, thickness, load, E, nu
    )
    return shear + bending


def apply_cantilever_clamp_lagrange(problem, plate, quadrature=None):
    condition = ck.LagrangeMultiplierCondition(
        boundary=plate.boundary("u0"),
        quadrature=quadrature,
        w_bar=0.0,
        phi_n_bar=0.0,
        phi_s_bar=0.0,
    )
    problem.add_condition(condition)


def apply_cantilever_clamp_penalty(problem, plate, quadrature=None):
    Kb = problem.element.material.bending_stiffness()
    alpha_w = PENALTY_SCALE * Kb / CANTILEVER_LENGTH**3
    alpha_phi = PENALTY_SCALE * Kb / CANTILEVER_LENGTH

    condition = ck.PenaltyCondition(
        plate.boundary("u0"),
        quadrature,
        alpha_w=alpha_w,
        w_bar=0.0,
        alpha_phi_n=alpha_phi,
        phi_n_bar=0.0,
        alpha_phi_s=alpha_phi,
        phi_s_bar=0.0,
    )
    problem.add_condition(condition)


def solve_cantilever_plate(
    problem,
    load_fn,
    bc_type,
    load_quadrature=None,
    boundary_quadrature=None,
):
    plate = problem.patches[0]
    load = ck.conditions.create_load_condition(
        patch=plate, load_fn=load_fn, quadrature=load_quadrature
    )
    problem.add_condition(load)

    K, F = problem.assemble()
    if bc_type == "penalty":
        apply_cantilever_clamp_penalty(problem, plate, boundary_quadrature)
    elif bc_type == "lagrange":
        apply_cantilever_clamp_lagrange(problem, plate, boundary_quadrature)
    else:
        raise ValueError("Invalid BC type")

    u = ck.solve(problem)
    return u, K, F


def centerline_displacement(problem, solution, x_values):
    plate = problem.patches[0]
    element = problem.element
    u = np.asarray(solution)[:problem.num_physical_dofs]

    xi = np.asarray(x_values, dtype=float) / CANTILEVER_LENGTH
    pts = np.column_stack([xi, 0.5 * np.ones_like(xi)])
    shape = ck.eval_shape_at(plate, pts, order=2)
    Nw = element._cpp_object.displacement_shape_matrix(shape)
    return np.asarray(Nw @ u)


cantilever_tip_results = {name: [] for name in FORMULATIONS}
cantilever_shear_fractions = []
cantilever_shape_results = {}

print("Cantilever plate under uniform transverse load")
print(
    f"Geometry: L = {CANTILEVER_LENGTH:g}, B = {CANTILEVER_WIDTH:g} "
    f"(B/L = {CANTILEVER_WIDTH / CANTILEVER_LENGTH:g}); clamp at X = 0"
)
print(
    "Boundary enforcement follows section 1.1: "
    f"Lagrange for L/h <= {THIN_PLATE_SWITCH:g}, penalty otherwise."
)
print(
    f"{'L/h':>10} | {'BC':>7} | {'w_s/w':>9} | {'Analytical tip':>15} | "
    f"{'KL-1':>10} | {'RM-3':>10} | {'RM-D3':>10} | {'RM-D2':>10} | {'RM-D1':>10}"
)
print("-" * 109)

for i, slenderness in enumerate(CANTILEVER_SLENDERNESS_VALUES):
    thickness = CANTILEVER_LENGTH / slenderness
    material = ck.material.create_plane_stress_2d(E, nu, thickness)
    bc_type = "penalty" if slenderness > THIN_PLATE_SWITCH else "lagrange"

    patch = ck.geometry.create_rectangle(
        nu=CANTILEVER_NBASIS_U,
        nv=CANTILEVER_NBASIS_V,
        deg=CANTILEVER_PDEG,
        l=CANTILEVER_LENGTH,
        w=CANTILEVER_WIDTH,
    )
    gauss2d_c = ck.GaussLegendre(CANTILEVER_PDEG + 1, dim=2)
    gauss1d_c = ck.GaussLegendre(CANTILEVER_PDEG + 1, dim=1)

    analytical_shear_tip, analytical_bending_tip = rm_beam_cantilever_components(
        CANTILEVER_LENGTH,
        CANTILEVER_LENGTH,
        CANTILEVER_WIDTH,
        thickness,
        CANTILEVER_LOAD,
        E,
        nu,
    )
    analytical_tip = analytical_shear_tip + analytical_bending_tip
    shear_fraction = analytical_shear_tip / analytical_tip
    cantilever_shear_fractions.append(shear_fraction)

    row_values = {}
    for formulation, element_cls in FORMULATIONS.items():
        element = element_cls(material)
        problem = ck.LinearElasticProblem([patch], element, gauss2d_c)
        u, _, _ = solve_cantilever_plate(
            problem, cantilever_load, bc_type, gauss2d_c, gauss1d_c
        )
        numerical_tip = abs(centerline_displacement(problem, u, [CANTILEVER_LENGTH])[0])
        ratio = numerical_tip / analytical_tip
        cantilever_tip_results[formulation].append(ratio)
        row_values[formulation] = f"{ratio:.3f}"

        if i == CANTILEVER_SHAPE_INDEX:
            x_line = np.linspace(0.0, CANTILEVER_LENGTH, 121)
            cantilever_shape_results[formulation] = abs(
                centerline_displacement(problem, u, x_line)
            )

    row = (
        f"{slenderness:10.2e} | {bc_type:>7} | {shear_fraction:9.2e} | "
        f"{analytical_tip:15.6e} | {row_values['KL-1']:>10} | "
        f"{row_values['RM-3']:>10} | {row_values['RM-D3']:>10} | "
        f"{row_values['RM-D2']:>10} | {row_values['RM-D1']:>10}"
    )
    print(row)

x_line = np.linspace(0.0, CANTILEVER_LENGTH, 121)
shape_thickness = CANTILEVER_LENGTH / CANTILEVER_SHAPE_SLENDERNESS
analytical_shape = rm_beam_cantilever_deflection(
    x_line,
    CANTILEVER_LENGTH,
    CANTILEVER_WIDTH,
    shape_thickness,
    CANTILEVER_LOAD,
    E,
    nu,
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

ax = axes[0]
for formulation in FORMULATIONS:
    ax.semilogx(
        CANTILEVER_SLENDERNESS_VALUES,
        cantilever_tip_results[formulation],
        marker="o",
        label=CANTILEVER_LEGEND_LABELS[formulation],
    )
ax.axhline(1.0, color="k", linestyle="--", alpha=0.35, label="RM beam reference")
ax.set_xlabel(r"Slenderness $L/h$")
ax.set_ylabel(r"Tip displacement ratio")
ax.set_title(r"Cantilever Free-Tip Response")
ax.grid(True, which="both", alpha=0.3)
ax.legend(fontsize=8)

ax = axes[1]
ax.plot(x_line / CANTILEVER_LENGTH, analytical_shape, "k--", linewidth=2.0, label="RM beam reference")
for formulation in FORMULATIONS:
    if formulation in cantilever_shape_results:
        ax.plot(
            x_line / CANTILEVER_LENGTH,
            cantilever_shape_results[formulation],
            linewidth=1.6,
            label=CANTILEVER_LEGEND_LABELS[formulation],
        )
ax.set_xlabel(r"$X/L$")
ax.set_ylabel(r"$|w(X,B/2)|$")
ax.set_title(rf"Centerline Shape at $L/h={CANTILEVER_SHAPE_SLENDERNESS:g}$")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)

plt.show()


### **RM-D2 Cantilever Deformation Fields**

The sweep above compares scalar response quantities. To see what the proposed RM-D2 formulation is doing in the cantilever setting, we now solve one representative RM-D2 case and recover the displacement decomposition. The surfaces below show the total transverse displacement $w$, the bending part $w_b$, and the shear contribution $w-w_b$. The in-plane axes are normalized by the cantilever length and width so the long, narrow plate remains readable.


In [ ]:
CANTILEVER_FIELD_SLENDERNESS = CANTILEVER_SHAPE_SLENDERNESS
field_thickness = CANTILEVER_LENGTH / CANTILEVER_FIELD_SLENDERNESS
field_bc_type = "penalty" if CANTILEVER_FIELD_SLENDERNESS > THIN_PLATE_SWITCH else "lagrange"

field_material = ck.material.create_plane_stress_2d(E, nu, field_thickness)
field_patch = ck.geometry.create_rectangle(
    nu=CANTILEVER_NBASIS_U,
    nv=CANTILEVER_NBASIS_V,
    deg=CANTILEVER_PDEG,
    l=CANTILEVER_LENGTH,
    w=CANTILEVER_WIDTH,
)
field_gauss2d = ck.GaussLegendre(CANTILEVER_PDEG + 1, dim=2)
field_gauss1d = ck.GaussLegendre(CANTILEVER_PDEG + 1, dim=1)
field_element = FORMULATIONS["RM-D2"](field_material)
field_problem = ck.LinearElasticProblem([field_patch], field_element, field_gauss2d)
field_u, _, _ = solve_cantilever_plate(
    field_problem,
    cantilever_load,
    field_bc_type,
    field_gauss2d,
    field_gauss1d,
)

n_x, n_y = 161, 41
u_grid = np.linspace(0.0, 1.0, n_x)
v_grid = np.linspace(0.0, 1.0, n_y)
uu, vv = np.meshgrid(u_grid, v_grid, indexing="xy")
pts = np.column_stack([uu.ravel(), vv.ravel()])
X = uu * CANTILEVER_LENGTH
Y = vv * CANTILEVER_WIDTH
Xn = X / CANTILEVER_LENGTH
Yn = Y / CANTILEVER_WIDTH

shape = ck.eval_shape_at(field_patch, pts, order=2)
field_u_phys = np.asarray(field_u)[:field_problem.num_physical_dofs]

Nw = field_element._cpp_object.displacement_shape_matrix(shape)
w_field = np.asarray(Nw @ field_u_phys).reshape(uu.shape)

wb_coeff = field_u_phys[0::2]
psi_coeff = field_u_phys[1::2]
wb_field = np.asarray(shape[0] @ wb_coeff).reshape(uu.shape)
psi_field = np.asarray(shape[0] @ psi_coeff).reshape(uu.shape)
psi_x = np.asarray(shape[1] @ psi_coeff).reshape(uu.shape)
psi_y = np.asarray(shape[2] @ psi_coeff).reshape(uu.shape)

ws_field = w_field - wb_field
curl_psi_x = psi_y
curl_psi_y = -psi_x
curl_psi_norm = np.sqrt(curl_psi_x**2 + curl_psi_y**2)

surface_fields = [
    (w_field, r"Total displacement $w$", r"$w$", "turbo"),
    (wb_field, r"Bending displacement $w_b$", r"$w_b$", "viridis"),
    (ws_field, r"Shear displacement $w-w_b$", r"$w-w_b$", "RdBu_r"),
]

fig = plt.figure(figsize=(16, 5), constrained_layout=True)
for i, (field, title, label, cmap) in enumerate(surface_fields, start=1):
    ax = fig.add_subplot(1, 3, i, projection="3d")
    surf = ax.plot_surface(
        Xn,
        Yn,
        field,
        cmap=cmap,
        linewidth=0,
        antialiased=True,
        rstride=2,
        cstride=1,
        alpha=0.95,
    )
    ax.contour(
        Xn,
        Yn,
        field,
        zdir="z",
        offset=float(np.min(field)),
        cmap=cmap,
        levels=14,
        linewidths=0.6,
    )
    ax.set_xlabel(r"$x/L$")
    ax.set_ylabel(r"$y/B$")
    ax.set_zlabel(label)
    ax.set_title(title)
    ax.view_init(elev=24, azim=-128)
    ax.set_box_aspect((3.0, 0.8, 0.7))
    fig.colorbar(surf, ax=ax, shrink=0.58, pad=0.04, label=label)

fig.suptitle(
    rf"RM-D2 cantilever displacement decomposition, $L/h={CANTILEVER_FIELD_SLENDERNESS:.2g}$"
)
plt.show()


### **Solenoidal Mode in the Cantilever**

The RM-D2 cantilever solution also contains the scalar potential $\psi$. Its curl, $\nabla\times\psi = [\partial\psi/\partial y, -\partial\psi/\partial x]^T$, is the solenoidal rotation contribution added to the curl-free displacement-gradient fields. The plot below shows both the recovered potential and the vector field it generates.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), constrained_layout=True)

ax = axes[0]
psi_min = float(np.min(psi_field))
psi_max = float(np.max(psi_field))
psi_levels = np.linspace(psi_min, psi_max, 31)
cf_psi = ax.contourf(Xn, Yn, psi_field, levels=psi_levels, cmap="viridis")
ax.contour(Xn, Yn, psi_field, levels=12, colors="k", linewidths=0.3, alpha=0.3)
ax.set_aspect("auto")
ax.set_xlabel(r"$x/L$")
ax.set_ylabel(r"$y/B$")
ax.set_title(r"Scalar potential $\psi$")
fig.colorbar(cf_psi, ax=ax, shrink=0.85, label=r"$\psi$")

ax = axes[1]
cf_curl = ax.contourf(Xn, Yn, curl_psi_norm, levels=31, cmap="turbo")
skip_x, skip_y = 8, 3
ax.quiver(
    Xn[::skip_y, ::skip_x],
    Yn[::skip_y, ::skip_x],
    curl_psi_x[::skip_y, ::skip_x],
    curl_psi_y[::skip_y, ::skip_x],
    color="white",
    pivot="mid",
    scale_units="xy",
    width=0.004,
)
ax.set_aspect("auto")
ax.set_xlabel(r"$x/L$")
ax.set_yticklabels([])
ax.set_title(r"Solenoidal rotation $\nabla\times\psi$")
fig.colorbar(cf_curl, ax=ax, shrink=0.85, label=r"$|\nabla\times\psi|$")

fig.suptitle(r"RM-D2 solenoidal mode in the cantilever benchmark")
plt.show()
